In [229]:
# This notebook aims to calculate the area of orthographic projections of our sample (in the .stl file) for varying sample angles, ψ.

using FileIO
using GeometryBasics
using MeshIO
using BenchmarkTools
using Distributions

In [72]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


# Dummy sample comprising an icosphere with 320 faces.
stl = load("STL_FileExamples/Icosphere320.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)
# Extracting the number of vertices.
const n_vert = length(vertices)

960

In [ ]:
# Defining a function to rotate the sample about the z-axis.

"""
Rotates the sample by θ degrees. This is accomplished by mutating each vertex.

Parameters
----------
θ (float): Rotation angle, in degrees.
vertices (vector with 3-vector elements with float elements): Vertices of sample, in units of .stl file.

Returns
-------
vertices (vector with 3-vector elements with float elements): Rotated vertices of sample, in units of .stl file.
"""
function rotate!(θ :: Float32, vertices :: Vector{Point{3, Float32}}) :: Vector{Point{3, Float32}}
    # Converting the angle to radians.
    θ = deg2rad(θ)
    # Calculating rotation matrix.
    R_z = Matrix{Float32}([[cos(θ), -sin(θ), 0] [sin(θ), cos(θ), 0] [0, 0, 1]])
    # Applying rotation matrix to each vertex.
    @inbounds for i in 1:n_vert
        vertices[i] = R_z * vertices[i]
    end
    return vertices
end

# @benchmark rotate!(30f0, vertices)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  35.200 μs …  18.068 ms  ┊ GC (min … max):  0.00% … 99.36%
 Time  (median):     42.500 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   62.710 μs ± 331.744 μs  ┊ GC (mean ± σ):  12.02% ±  2.43%

   ▅▇█▆                                                         
  ▆█████▄▂▂▂▂▂▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  35.2 μs         Histogram: frequency by time          129 μs <

 Memory estimate: 76.12 KiB, allocs estimate: 1952.

In [225]:
# Projecting the sample onto the y-z plane (the plane perpendicular to the neutron beam).
# This is accomplished by simply reading off the y and z coordinates of the vertices.

p_vertices = Vector{Vector{Float32}}(undef, n_vert)
for i in 1:n_vert
    p_vertices[i] = [vertices[i][2], vertices[i][3]]
end
p_vertices

960-element Vector{Vector{Float32}}:
 [0.95625895, -0.24883369]
 [0.95100224, 0.010683945]
 [1.0, 0.0]
 [-0.86402386, -0.24883369]
 [-0.95100224, 0.010683945]
 [-0.809017, 0.0]
 [0.95625895, 0.24883369]
 [0.95100224, -0.010683945]
 [1.0, 0.0]
 [0.29252154, 0.5027351]
 ⋮
 [0.86402386, 0.5027351]
 [0.7147844, 0.65965474]
 [0.8506508, 0.4472136]
 [0.6825348, 0.7283435]
 [0.86402386, 0.5027351]
 [0.809017, 0.5257311]
 [0.86402386, 0.5027351]
 [0.6825348, 0.7283435]
 [0.7147844, 0.65965474]

In [230]:
# Grouping these projected vertices by the triangles they create.

p_triangles = Vector{Vector{Vector[Float32]}}(undef, n_faces)
for i in 1:n_faces
    p_triangles[i] = [p_vertices[indices[i][1]], p_vertices[indices[i][2]], p_vertices[indices[i][3]]]
end

MethodError: MethodError: Cannot `convert` an object of type 
  Type{Float32} to an object of type 
  Vector
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  convert(::Type{Vector}, !Matched::StatsBase.UnitWeights{T}) where T
   @ StatsBase C:\Users\qfp51584\.julia\packages\StatsBase\s2YJR\src\weights.jl:321
  convert(::Type{Vector}, !Matched::StatsBase.AbstractWeights)
   @ StatsBase C:\Users\qfp51584\.julia\packages\StatsBase\s2YJR\src\weights.jl:35
  convert(::Type{T}, !Matched::AbstractArray) where T<:Array
   @ Base array.jl:618
  ...


In [ ]:
# Finding the maximum and minimum value of each projected coordinate.
# This is used to create a 2D bounding rectangle.

max_coord = [maximum(getindex.(p_vertices, 1)), maximum(getindex.(p_vertices, 2))]
min_coord = [minimum(getindex.(p_vertices, 1)), minimum(getindex.(p_vertices, 2))]
# Calculating the area of the bounding rectangle.
rec_area = (max_coord[1] - min_coord[1]) * (max_coord[2] - min_coord[2])

4.0f0

In [ ]:
# Setting the total number of random coordinates to be generated within the bounding rectangle.

const n_tot = 1000

In [ ]:
# Defining the function to calculate the area, in units of the .stl file squared, of the sample projected into the y-z plane.

function area_calc()
    # Tallying the number of random coordinates that lie within the sample.
    n_in = 0
    # Pre-calculating the uniform distribution describing the bounding rectangle.
    y_range = Uniform(min_coord[1], max_coord[1])
    z_range = Uniform(min_coord[2], max_coord[2])
    for i in 1:n_tot
        # Generating a random 2D coordinate within the pre-defined ranges.
        y = rand(y_range)
        z = rand(z_range)
        for j in 1:n_faces
            triangle = 
            # In barycentric coordinates, any point in a triangle can be expressed as (1-u-v) * V1 + u * V2 + v * V3 where u, v ≥ 0 and u + v ≤ 1.
            


